# 02 — Regression and GDP growth regimes

This notebook fits the log GDP trend regression and the piecewise constant growth-regime model.

In [ ]:
from pathlib import Path
import pandas as pd

from us_gdp_regime.config import load_config
from us_gdp_regime.models import assign_segment_labels, fit_growth_regimes, fit_log_trend
from us_gdp_regime.plotting import plot_growth_regimes, plot_log_gdp_trend
from us_gdp_regime.pipeline import _load_primary_series
from us_gdp_regime.transform import validate_gdp_series

config = load_config(Path('..') / 'config' / 'default.yaml')
series = validate_gdp_series(_load_primary_series(config))


## Log real GDP trend

GDP levels are non-stationary. The log trend gives a compact estimate of the long-run average growth path.

In [ ]:
trend_result, trend_frame = fit_log_trend(series)
trend_result


In [ ]:
plot_log_gdp_trend(
    series=series,
    trend_frame=trend_frame,
    trend_result=trend_result,
    output_path=Path('..') / 'figures' / 'log_real_gdp_trend.png',
    dpi=config.plots.dpi,
)


## Piecewise growth regimes

Above/below-mean labels are assigned to growth regimes, not GDP levels. This avoids mechanically classifying later years as above mean simply because GDP has trended upward.

In [ ]:
segments, segment_frame = fit_growth_regimes(
    series,
    min_segment_size=config.model.min_segment_size,
    max_breaks=config.model.max_breaks,
    criterion=config.model.criterion,
)
segment_frame


In [ ]:
labelled = assign_segment_labels(series, segments)
plot_growth_regimes(
    series=labelled,
    segments=segments,
    output_path=Path('..') / 'figures' / 'gdp_growth_regimes.png',
    dpi=config.plots.dpi,
)


## Sensitivity skeleton

This block compares breakpoints across a few settings. A full robustness module can be added later using the prompts in `prompts/01_code_development.md`.

In [ ]:
rows = []
for criterion in ['bic', 'aic']:
    for min_size in [4, 5, 7, 10]:
        try:
            _, frame = fit_growth_regimes(series, min_segment_size=min_size, max_breaks=8, criterion=criterion)
            frame = frame.assign(criterion=criterion, min_segment_size=min_size)
            rows.append(frame)
        except ValueError as exc:
            print(f'Skipped criterion={criterion}, min_size={min_size}: {exc}')

sensitivity = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
sensitivity


In [ ]:
models_dir = Path('..') / 'data' / 'models'
models_dir.mkdir(parents=True, exist_ok=True)
pd.DataFrame([trend_result.__dict__]).to_csv(models_dir / 'trend_regression.csv', index=False)
segment_frame.to_csv(models_dir / 'regime_segments.csv', index=False)
if not sensitivity.empty:
    sensitivity.to_csv(models_dir / 'regime_sensitivity.csv', index=False)
